In [1]:
import pandas as pd


In [3]:
data = pd.read_parquet('/Users/chankyulee/Desktop/ModuLABS/08_HACKATHONS/AIFFELTHON_SIA/gdelt_main_0331.parquet')
data.head(5)

,GLOBALEVENTID,SQLDATE,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,QuadClass,...,NumMentions,NumArticles,NumSources,AvgTone,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,ActionGeo_Type
0,1281099176,20251226,UNITED STATES,USA,None,None,0,10.0,1.0,1,...,4,4,1,-2.895323,Ghouta,SY,33.500000,36.383301,-2538031,4
1,1281094432,20251226,ISRAEL,ISR,None,None,0,128.0,12.0,3,...,1,1,1,-3.092783,Golan Heights,SY,32.904900,35.807201,-2538190,4
2,1280985542,20251226,ISRAEL,ISR,None,None,0,120.0,12.0,3,...,4,4,1,-6.399132,Jordan Valley,IS,32.666698,35.500000,-2542918,4
3,1281098790,20251226,ISRAELI,ISR,None,None,0,90.0,9.0,2,...,6,6,1,-8.682171,Jordan Valley,IS,32.666698,35.500000,-2542918,4
4,1281049155,20251226,MARYLAND,USA,None,None,0,43.0,4.0,1,...,8,8,4,-3.381267,Alexandria,EG,31.198099,29.919201,-290263,4


In [4]:
data_url = pd.read_parquet('/Users/chankyulee/Desktop/ModuLABS/08_HACKATHONS/AIFFELTHON_SIA/gdelt_url_0331.parquet')
data_url.head(5)

,GLOBALEVENTID,SOURCEURL
0,1276719967,https://timesofoman.com/article/165580-israeli...
1,1276722901,https://dunyanews.tv/en/World/920726-israeli-f...
2,1276781023,https://www.seattletimes.com/nation-world/isra...
3,1276733974,https://english.aawsat.com/arab-world/5214015-...
4,1276801424,https://www.pilotonline.com/2025/11/29/william...


In [ ]:
# 3월 31일 사건이 업데이트 되었다고 치면..

import pandas as pd
import numpy as np
from pipeline.config import tone_weight, CONFIRMED_CODES, TRIAD_COUNTRIES
from pipeline.conflict_index import compute_conflict_index, kalman_innovation

# 1. 데이터 로드 및 통합 (기존 데이터 + 업로드한 3월 31일 데이터)
combined_data = data

# 2. 갈등 지수(I) 산출 
# NumMentions와 AvgTone 가중치를 결합하여 실제 사건의 파급력을 지수화합니다.
print("--- 갈등 지수(I) 산출 중 ---")
city_daily = compute_conflict_index(combined_data)

# 3. 칼만 필터 기반 이상탐지 (롤링 윈도우 제외 버전)
# 필터 내부의 예측 오차(Innovation)를 필터가 추정한 표준편차(sqrt(S))로 나눈 
# '정규화된 혁신(Normalized Innovation)'을 직접 Z-Score로 사용합니다.
results = []
target_date = "20260331"

for city, group in city_daily.groupby('city'):
    group = group.sort_values('date').reset_index(drop=True)
    signal = group['conflict_index'].values.astype(float)
    
    if len(signal) < 2: continue
    
    # 칼만 필터 실행 (내부에서 Q, R 자동 추정)
    kf = kalman_innovation(signal)
    
    # 롤링 윈도우 없이 필터가 계산한 norm_innov(Normalized Innovation)를 그대로 사용
    group['kalman_z'] = kf['norm_innov']
    group['is_anomaly'] = group['kalman_z'] > 2.0 # 임계치 2.0 적용
    
    results.append(group)

# 4. 결과 출력
final_df = pd.concat(results)
today_result = final_df[final_df['date'] == target_date].sort_values('kalman_z', ascending=False)

print(f"\n📢 {target_date} 이상 징후 분석 결과 (Kalman Innovation Z):")
print("-" * 80)
if not today_result[today_result['is_anomaly']].empty:
    for _, row in today_result[today_result['is_anomaly']].iterrows():
        print(f"🚨 {row['city']:15s} | 혁신 Z: {row['kalman_z']:>5.2f} | 갈등지수 I: {row['conflict_index']:>8.1f}")
else:
    print("✅ 3월 31일 감지된 이상 도시가 없습니다.")

# 상위 5개 도시 현황 출력
print("\n📊 상위 갈등 지수 도시 현황:")
print(today_result[['city', 'conflict_index', 'kalman_z', 'events']].head(5))
